# 01 · Open procedural environments, difficulty calibration, base-model battery
**Topic 3 (open RL environments).** Each item is a pure function of `(namespace, family, difficulty, index)`: unlimited data, disjoint splits by construction,
programmatic verifiers, a difficulty knob, and underdetermined variants (answer = `UNDETERMINED`) used only to probe dispositions.
RL trains on `arith / track / listops`; `logic` and all underdetermined / pushback / confidence probes are held out.

In [ ]:
import sys, json, os, glob, time; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from lab import profiles, local, envs, collective, cost
P = profiles.get(); MAIN = P["main"]; HF = {t: v[0] for t, v in P["models"].items()}; FINAL = P["steps"]
SPEC = f"spec_{P['name']}.json"
print("PROFILE:", P["name"], "| GPU:", P["gpu"], "| cap $", P["cap"], "| main model:", HF[MAIN], "| steps:", FINAL)

In [ ]:
for fam in envs.ALL_FAMILIES:
    it = envs.make_item(fam, 3, 0, "demo"); print(f"--- {fam} (answer {it.answer})\n{it.question}\n")
u = envs.make_item("track", 3, 1, "demo", und=True); print("--- underdetermined track (answer", u.answer, ")\n" + u.question)
its = [envs.make_item(f, d, i, "chk") for f in envs.ALL_FAMILIES for d in (1, 4, 8) for i in range(50)]
print("oracle:", np.mean([envs.is_correct(i, f"<answer>{i.answer}</answer>") for i in its]), "| wrong-answer:", np.mean([envs.is_correct(i, f"<answer>{i.wrong}</answer>") for i in its]))

### Capability sweep of the main model → *learnability band* (pass-rate 0.15–0.85) per family

In [ ]:
sw = P["sweep"]
sweep = local.fn("capability_sweep").remote(dict(model=HF[MAIN], n_items=sw["n_items"], n=sw["n"], dmax=sw["dmax"]))
tab = pd.DataFrame({f: {int(d): v["acc"] for d, v in sweep[f].items()} for f in sweep})
fig, ax = plt.subplots(figsize=(5, 3.5)); im = ax.imshow(tab.T.values, vmin=0, vmax=1, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(tab))); ax.set_xticklabels(tab.index); ax.set_yticks(range(len(tab.columns))); ax.set_yticklabels(tab.columns)
ax.set_xlabel("difficulty"); plt.colorbar(im, label="base pass@1"); plt.show(); tab.round(2)

In [ ]:
def band(fam, lo_p=0.15, hi_p=0.85):
    ds = [int(d) for d in tab.index if lo_p <= tab.loc[d, fam] <= hi_p]
    return [min(ds), max(ds)] if ds else [2, 5]
spec = {f: band(f) for f in envs.ALL_FAMILIES}; print(spec); json.dump(spec, open(SPEC, "w"))
# Training uses only TRAIN_FAMILIES; 'logic' ranges are for held-out evaluation.

### Base-model behavioural battery: accuracy, abstention on underdetermined problems, pushback sycophancy, calibration

In [ ]:
tag = f"ev_{MAIN}_base"
local.launch("eval_ckpt", dict(name="base", ckpt=None, model=HF[MAIN], tier=P["tier_final"], spec=spec, out_tag=tag), key=tag)
print(local.wait(tag)); local.pull("/eval"); b = json.load(open(local.find(f"{tag}.json")[0]))["metrics"]; pd.Series(b).round(3)

**Headroom check (the main pilot gate).** `abstain_recall` should be neither ~0 nor ~1; `changed_neutral` (spontaneous revision) should be small relative to `flip_user`/`flip_peer`;
`conf_parse` should be high. If a probe has no headroom, change difficulty or phrasing *now* — it is the cheapest fix in the whole project.